# sales_line — Integration view

Geïntegreerde business-view op **line-grain**: `INTEGRATION.DWH_ORDER_DETAIL ⋈ INTEGRATION.DWH_ORDER_HEADER` op `order_id`. Eén rij per `order_detail_id`, met alle header-attributen gedenormaliseerd op iedere regel.

**SCD2 join semantics:** de header-versie die wordt gekozen, is de versie die **geldig was op `order_ts`** (event time van de regel). Dit gebeurt via het half-open interval `WA_FROMDATE <= order_ts < WA_UNTODATE` op `DWH_ORDER_HEADER`. Voor de lines zelf nemen we de huidige versie (`WA_ISCURR = 1`), omdat lines in deze demo niet inhoudelijk versioneren.

**Waarom een view (geen MV):** geen state nodig, pure join — Spark optimizer pusht filters door naar beide DWH-views. Header-correcties (nieuwe SCD2-versies) verschijnen direct, geen refresh-lag.

Georkestreerd door `views/07_apply_views.ipynb` (`dbutils.notebook.run`).

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";

CREATE OR REPLACE VIEW ${catalog}.INTEGRATION.SALES_LINE AS
SELECT
  d.WK_ORDER_DETAIL,
  d.WKR_ORDER_DETAIL,
  h.WK_ORDER_HEADER,
  h.WKR_ORDER_HEADER,
  -- line attrs
  d.order_detail_id,
  d.order_id,
  d.menu_item_id,
  d.quantity,
  d.unit_price,
  d.price,
  d.order_item_discount_amount,
  d.line_number,
  -- header attrs (from header version current at the line's event time)
  h.truck_id,
  h.location_id,
  h.customer_id,
  h.shift_id,
  h.discount_id,
  h.shift_start_time,
  h.shift_end_time,
  h.order_channel,
  h.order_ts,
  h.served_ts,
  h.order_currency,
  h.order_amount,
  h.order_tax_amount,
  h.order_discount_amount,
  h.order_total,
  -- admin (line's)
  d.WA_CRUDDTS,
  d.WA_CRUD,
  d.WA_SRC,
  d.WA_RUNID
FROM ${catalog}.INTEGRATION.DWH_ORDER_DETAIL d
JOIN ${catalog}.INTEGRATION.DWH_ORDER_HEADER h
  ON d.order_id = h.order_id
  AND h.WA_FROMDATE <= h.order_ts
  AND h.order_ts < h.WA_UNTODATE
WHERE d.WA_ISCURR = 1